# Aula 1 — Um pedido, três papéis com LLM

Vamos passar o pedido P001 por estoque, logística e revisão. Ana, Bruno e Carla usam o mesmo modelo, cada um com seu prompt e seu histórico. Python define a ordem das etapas; o modelo gera as respostas.

A multiplicação continua em uma função pequena, como no Agente matemático. O modelo chama essa ferramenta, e nós conferimos o valor que ela devolveu. Todas as propostas da loja são fictícias.

## Preparar o Colab

Execute a célula e espere a instalação terminar. Depois, siga para o passo 1. As versões são fixas e vêm do PyPI oficial. Se o próprio Colab pedir reinício, use **Runtime → Restart session** e execute desde o início. Use uma sessão CPU; o modelo será acessado na nuvem do Ollama.

In [1]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
REQUISITOS = {'langchain': '1.4.7', 'langchain-core': '1.7.1', 'pydantic': '2.13.5', 'langchain-ollama': '1.1.0', 'langgraph': '1.2.18', 'ollama': '0.6.3'}
def versao_instalada(nome):
    try: return version(nome)
    except PackageNotFoundError: return None
faltantes = [f'{p}=={v}' for p, v in REQUISITOS.items() if versao_instalada(p) != v]
if IN_COLAB and faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '--disable-pip-version-check', '--index-url', 'https://pypi.org/simple', *faltantes], check=True)
elif faltantes: raise RuntimeError('Fora do Colab não instalamos automaticamente. Use o ambiente preparado nas versões indicadas no README.')
print('Python do kernel:', sys.executable, '| Colab:', IN_COLAB)
print('Versões conferidas:', {p: versao_instalada(p) for p in REQUISITOS})

RuntimeError: Fora do Colab não instalamos automaticamente. Use o ambiente preparado nas versões indicadas no README.

## Antes de começar

No cadeado do Colab, confira o segredo chamado **segredo**, com sua chave do Ollama. Autorize o acesso deste notebook. A chave não deve ser colada nas células nem aparecer nas saídas.

Retomamos o P001 do Módulo 3: duas camisetas a R$ 35 e estoque inicial de oito. Os dados estão preparados neste caderno. Não há pagamento, reserva ou envio real.

## 1. Conectar o modelo

Esta célula importa o LangChain, lê o segredo e configura o Ollama. A primeira pergunta será enviada no passo 3. Não é necessário instalar o Ollama no computador nem baixar o modelo.

In [ ]:
import os, json
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['OTEL_SDK_DISABLED'] = 'true'
from google.colab import userdata
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
chave = userdata.get('segredo')
modelo = ChatOllama(model='qwen3.5:4b', base_url='https://localhost',
    temperature=1, num_predict=256, validate_model_on_init=False,
    client_kwargs={'headers': {'Authorization': 'Bearer ' + chave}, 'timeout': 60})
print('Modelo configurado. A chave não será exibida.')

Modelo configurado. A chave não será exibida.


## 2. Definir a ferramenta, os dados e os papéis

A ferramenta faz a conta em Python. A referência 6×7 continua sendo 42; o subtotal das camisetas é outra conta. Depois criamos os três agentes. Ana recebe a ferramenta; Bruno e Carla trabalham com os pareceres que chegam.

O estoque inicial faz parte do contexto enviado ao modelo. Os agentes escrevem propostas em texto; este exemplo não altera um sistema de estoque.

In [ ]:
@tool
def multiplicar(a: int, b: int) -> int:
    """Multiplica dois números e devolve o resultado exato."""
    return a * b

pedido = {'id': 'P001', 'item': 'camiseta', 'quantidade': 2}
preco_unitario = 35
estoque_inicial = 8
referencia = multiplicar.invoke({'a': 6, 'b': 7})
contexto = json.dumps({'pedido': pedido, 'preco_unitario': preco_unitario,
                      'estoque_inicial': estoque_inicial}, ensure_ascii=False)
print('Dados preparados:', contexto)

Dados preparados: {"pedido": {"id": "P001", "item": "camiseta", "quantidade": 2}, "preco_unitario": 35, "estoque_inicial": 8}


In [ ]:
ana = create_agent(modelo, tools=[multiplicar], system_prompt=(
    'Você é Ana, do estoque. Use multiplicar para calcular o subtotal. '
    'Se a quantidade for positiva e couber no estoque, proponha reserva e mostre o restante. '
    'Caso contrário, informe SEM_ESTOQUE. Responda em português, em até três linhas.'))
bruno = create_agent(modelo, tools=[], system_prompt=(
    'Você é Bruno, da logística. O parecer de Ana é uma proposta fictícia, sem confirmação real. '
    'Com reserva proposta, escreva uma proposta de embalar; sem estoque, recuse a liberação. '
    'Não diga reserva confirmada nem afirme execução. Responda em português, em até três linhas.'))
carla = create_agent(modelo, tools=[], system_prompt=(
    'Você é Carla, revisora. Confira a conta, o estoque e a proposta de Bruno. '
    'Confira se alguma proposta foi tratada como confirmação. Aponte erros e lembre o caráter fictício. '
    'Responda em português, em até três linhas.'))

## 3. Ana consulta a ferramenta

Preveja o subtotal e o estoque restante. O agente recebe os dados e pode chamar multiplicar. A resposta guarda as mensagens do modelo e da ferramenta.

In [ ]:
resposta_ana = ana.invoke({'messages': [{'role': 'user', 'content': contexto}]},
                         config={'recursion_limit': 8})
texto_ana = resposta_ana['messages'][-1].content
print('Ana — estoque:', texto_ana)

Ana — estoque: Subtotal: R$ 70.
Reserva de 2 camisetas proposta.
Restante no estoque: 6.


## 4. Bruno recebe o parecer de Ana

O texto de Ana entra explicitamente na próxima mensagem. Observe se Bruno respeita a situação do estoque.

In [ ]:
mensagem_bruno = contexto + '\nParecer de Ana:\n' + texto_ana
resposta_bruno = bruno.invoke({'messages': [{'role': 'user', 'content': mensagem_bruno}]},
                             config={'recursion_limit': 8})
texto_bruno = resposta_bruno['messages'][-1].content
print('Bruno — logística:', texto_bruno)

Bruno — logística: Olá, aqui é o Bruno da logística. Com a reserva proposta por Ana, podemos planejar a embalagem dos itens. No aguardo de confirmações.


## 5. Carla revisa as duas respostas

Carla recebe o pedido e os dois pareceres. Leia a revisão antes de olhar a conferência em Python: ela identificou os valores e alguma inconsistência?

In [ ]:
mensagem_carla = (contexto + '\nParecer de Ana:\n' + texto_ana
                 + '\nParecer de Bruno:\n' + texto_bruno)
resposta_carla = carla.invoke({'messages': [{'role': 'user', 'content': mensagem_carla}]},
                             config={'recursion_limit': 8})
print('Carla — revisão:', resposta_carla['messages'][-1].content)

Carla — revisão: Olá, sou a Carla. A conta e o estoque estão corretos, mas Bruno tratou a proposta de Ana como confirmação ao planejar a embalagem. Lembro que os dados são fictícios.


## Conferir o que foi executado

A conferência abaixo usa a mensagem da ferramenta, e não uma conta copiada da resposta do modelo. Para o caso-base, esperamos subtotal 70 e referência 42. Os nomes de modelo nas três etapas mostram a origem das respostas.

Compare também os textos: o estoque esperado é 6, e embalar continua sendo uma proposta. A conferência da multiplicação não prova que todo o texto dos agentes está correto.
Sem estoque suficiente, a conferência exige SEM_ESTOQUE no parecer e permite que Ana dispense a conta. O número de ferramentas observadas fica visível.


In [ ]:
ferramentas = [m for m in resposta_ana['messages'] if isinstance(m, ToolMessage)]
assert all(m.status == 'success' for m in ferramentas), 'Confira a ferramenta de Ana.'
if 0 < pedido['quantidade'] <= estoque_inicial:
    assert ferramentas, 'Ana precisa calcular o subtotal com a ferramenta.'
    subtotal_observado = float(ferramentas[-1].content)
    assert subtotal_observado == pedido['quantidade'] * preco_unitario, 'O subtotal não confere.'
    print('Ferramenta:', ferramentas[-1].name, '→', subtotal_observado)
else:
    assert 'SEM_ESTOQUE' in texto_ana.upper(), 'Confira o parecer de estoque.'
    print('Sem estoque suficiente; não exigimos cálculo. Ferramentas:', len(ferramentas))
print('Referência 6×7:', referencia)
for nome, etapa in [('Ana', resposta_ana), ('Bruno', resposta_bruno), ('Carla', resposta_carla)]:
    print(nome, 'usou:', etapa['messages'][-1].response_metadata.get('model'))


Ferramenta: multiplicar → 70.0
Referência 6×7: 42
Ana usou: gemma4:31b
Bruno usou: gemma4:31b
Carla usou: gemma4:31b


## Sua vez

Tente criar um sistema multiagente próprio. Utilize alguma das arquiteturas ensinadas.

## Para consultar

- [Agentes com create_agent — LangChain](https://docs.langchain.com/oss/python/langchain/agents)
- [ChatOllama — LangChain](https://docs.langchain.com/oss/python/integrations/chat/ollama)
- [API na nuvem — Ollama](https://docs.ollama.com/cloud)

Fontes conferidas em 29/10/2026. A execução depende da chave, do acesso ao modelo e da disponibilidade do Ollama.